In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

In [5]:
df = pd.read_csv("CreditCardTransaction.csv")

In [6]:
from sklearn.ensemble import IsolationForest

In [7]:
features = df[['TrnxAmount']]

In [8]:
model = IsolationForest(
    contamination=0.01,
    random_state=42
)

df['anomaly'] = model.fit_predict(features)

In [9]:
df['anomaly'] = df['anomaly'].map({
    1:0,
    -1:1
})

In [10]:
df['anomaly'].value_counts()

anomaly
0    780692
1      7829
Name: count, dtype: int64

In [12]:
df[df['anomaly']==1][
    ['Department','Merchant','TrnxAmount']
].head(20)

,Department,Merchant,TrnxAmount
246,JUDICIAL,CAN*CANONFINANCIAL CFS,5707.87
484,EXECUTIVE,BOULEVARD AUTO SALES,5834.27
505,EXECUTIVE,FRED DRAKE AUTOMOTIVE,14765.92
533,EXECUTIVE,CHEVROLET OF DOVER II,8923.21
544,EXECUTIVE,DIAMOND STATE TIRE,6023.18
554,EXECUTIVE,WINNER PREMIER COLLISION,11146.92
561,EXECUTIVE,E ZPASS DE CSC00100701,18575.00
563,EXECUTIVE,B And G AUTO BODY,9640.13
567,EXECUTIVE,BOULEVARD AUTO SALES,5710.03
782,DEPT OF TECHNOLOGY AND INFOR,NASCIO,9000.00


In [13]:
#merchant frequency
merchant_freq = df['Merchant'].value_counts()

df['MerchantFreq'] = df['Merchant'].map(
    merchant_freq
)

In [15]:
#department frequency
dept_freq = df['Department'].value_counts()

df['DeptFreq'] = df['Department'].map(
    dept_freq
)

In [20]:
df['TranxDate'] = pd.to_datetime(
    df['TranxDate'],
    errors='coerce'
)
df['TranxDate'].dtype

dtype('<M8[ns]')

In [21]:
df['TranxDate'].isna().sum()

np.int64(324016)

In [22]:
df['Weekday'] = df['TranxDate'].dt.dayofweek

In [23]:
df['Month_Num'] = df['TranxDate'].dt.month

In [24]:
df['TranxDate'].head()

0   2018-06-29
1          NaT
2          NaT
3   2018-06-30
4   2018-06-27
Name: TranxDate, dtype: datetime64[ns]

In [25]:
df['TranxDate'].dtype

dtype('<M8[ns]')

In [26]:
df['TranxDate'].isna().sum()

np.int64(324016)

In [27]:
raw_df = pd.read_csv("CreditCardTransaction.csv")

In [28]:
bad_dates = raw_df.loc[
    df['TranxDate'].isna(),
    'TranxDate'
]

bad_dates.head(20)

1      07-03-2018
2      07-01-2018
7      07-04-2018
12     07-12-2018
13     07-12-2018
17     07-07-2018
18     07-12-2018
34     07-11-2018
39     07-12-2018
42     07-07-2018
43     07-07-2018
44     07-04-2018
76     07-10-2018
77     07-11-2018
81     07-01-2018
82     07-10-2018
85     07-11-2018
99     07-12-2018
100    07-02-2018
103    07-12-2018
Name: TranxDate, dtype: object

In [29]:
df['TranxDate'] = pd.to_datetime(
    raw_df['TranxDate'],
    format='mixed',
    errors='coerce'
)

In [30]:
df['TranxDate'].isna().sum()

np.int64(0)

In [31]:
df['TranxDate'].isna().sum()

np.int64(0)

In [32]:
merchant_freq = df['Merchant'].value_counts()

df['MerchantFreq'] = df['Merchant'].map(merchant_freq)

In [33]:
dept_freq = df['Department'].value_counts()

df['DeptFreq'] = df['Department'].map(dept_freq)

In [34]:
features = df[
    [
        'TrnxAmount',
        'MerchantFreq',
        'DeptFreq',
        'Weekday',
        'Month_Num'
    ]
]

In [35]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(features)

In [36]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    contamination=0.005,
    random_state=42
)

pred = model.fit_predict(X_scaled)

df['anomaly_v2'] = (pred == -1).astype(int)

In [37]:
df['anomaly_v2'].value_counts()

anomaly_v2
0    784579
1      3942
Name: count, dtype: int64

In [38]:
df[df['anomaly_v2']==1][
    ['Department',
     'Merchant',
     'TrnxAmount',
     'MerchantFreq',
     'DeptFreq']
].head(20)

,Department,Merchant,TrnxAmount,MerchantFreq,DeptFreq
196,JUDICIAL,CAN*CANONFINANCIAL CFS,4395.53,9247,19194
246,JUDICIAL,CAN*CANONFINANCIAL CFS,5707.87,9247,19194
833,DEPT OF TECHNOLOGY AND INFOR,CARAHSOFT TECHNOLOGY CORP,79306.34,19,6009
1065,STATE TREASURER,GRAINGER,43.63,16526,992
2289,DEPT OF FINANCE,GRAINGER,93.78,16526,5512
3416,SVS FR CHILDREN YOUTH FAMILIES,DMI* DELL HLTHCR/REL,26940.40,3629,38811
4755,DEPT OF CORRECTIONS,WB MASON,22141.20,3830,88181
4930,DEPT OF CORRECTIONS,THE CLASS PRODUCE GROUP,20608.64,237,88181
4953,DEPT OF CORRECTIONS,STATE JANITORAL SUPPLY,10407.70,5196,88181
4980,DEPT OF CORRECTIONS,STATE JANITORAL SUPPLY,15806.00,5196,88181


In [39]:
df['anomaly_score'] = model.decision_function(
    X_scaled
)

In [40]:
df['anomaly_score'].describe()

count    788521.000000
mean          0.132858
std           0.034485
min          -0.076495
25%           0.118401
50%           0.141392
75%           0.157460
max           0.176400
Name: anomaly_score, dtype: float64

In [41]:
df['risk_level'] = pd.cut(
    df['anomaly_score'],
    bins=[
        -1,
        -0.05,
        0,
        0.05,
        1
    ],
    labels=[
        'Critical',
        'High',
        'Medium',
        'Low'
    ]
)

In [42]:
dept_risk = (
    df.groupby('Department')
      .agg(
          Total_Transactions=('anomaly_v2','count'),
          Anomalies=('anomaly_v2','sum')
      )
)

In [43]:
dept_risk['Anomaly_Rate'] = (
    dept_risk['Anomalies']
    /
    dept_risk['Total_Transactions']
    *100
)

In [44]:
dept_risk.sort_values(
    'Anomaly_Rate',
    ascending=False
).head(20)

,Total_Transactions,Anomalies,Anomaly_Rate
Department,,,
STATE TREASURER,992,50,5.040323
DEPT OF TECHNOLOGY AND INFOR,6009,153,2.546181
DEPARTMENT OF HUMAN RESOURCES,2291,56,2.444347
DEPT OF CORRECTIONS,88181,1165,1.321146
LAS AMERICAS ASPIRA,3578,35,0.978200
EXECUTIVE,17138,163,0.951103
DEPT OF TRANSPORTATION,85203,768,0.901377
JUDICIAL,19194,145,0.755444
FIRE PREVENTION COMMISSION,3626,22,0.606729


In [45]:
#Top Departments by Anomaly Rate
dept_risk.sort_values(
    'Anomaly_Rate',
    ascending=False
).head(15)

,Total_Transactions,Anomalies,Anomaly_Rate
Department,,,
STATE TREASURER,992,50,5.040323
DEPT OF TECHNOLOGY AND INFOR,6009,153,2.546181
DEPARTMENT OF HUMAN RESOURCES,2291,56,2.444347
DEPT OF CORRECTIONS,88181,1165,1.321146
LAS AMERICAS ASPIRA,3578,35,0.978200
EXECUTIVE,17138,163,0.951103
DEPT OF TRANSPORTATION,85203,768,0.901377
JUDICIAL,19194,145,0.755444
FIRE PREVENTION COMMISSION,3626,22,0.606729


In [46]:
#Top Merchants by Number of Anomalies
df[df['anomaly_v2']==1]\
.groupby('Merchant')\
.size()\
.sort_values(ascending=False)\
.head(20)

Merchant
GRAINGER                    2631
CAN*CANONFINANCIAL CFS       232
VERIZONWRLSS*RTCCR VB        147
CITY OF WILMINGTON            87
VERIZON*ONETIMEPAYMENT        82
H. SCHRIER And CO. INC.       56
DMI* DELL HLTHCR/REL          54
WASTE MGMT WM EZPAY           45
STATE JANITORAL SUPPLY        31
CITY OF WILM-DIV REVENUE      27
KARETAS FOODS                 20
BARR INTERNATIONAL            20
SCHMIDT BAKING CO.            19
GOOD SOURCE SOLUTIONS         18
VZWRLSS*IVR VB                18
HYPOINT DAIRY FARMS INC       18
THE CLASS PRODUCE GROUP       16
MTM TECHNOLOGIES INC          16
WB MASON                      14
ATLANTIC BEVERAGE             13
dtype: int64

In [47]:
grainger = df[df['Merchant']=="GRAINGER"]

grainger['TrnxAmount'].describe()

count    16526.000000
mean       257.235845
std        657.661276
min     -15547.160000
25%         34.112500
50%         90.930000
75%        247.637500
max      19986.440000
Name: TrnxAmount, dtype: float64

In [48]:
df.groupby('Year')['TrnxAmount'].sum()

Year
2019    73886994.50
2020    70939105.50
2021    68059922.62
2022    77081142.78
2023    48017420.30
Name: TrnxAmount, dtype: float64

In [49]:
df.groupby('Year')['anomaly_v2'].sum()

Year
2019    739
2020    951
2021    938
2022    999
2023    315
Name: anomaly_v2, dtype: int64

In [50]:
year_stats = df.groupby('Year').agg(
    Total_Spend=('TrnxAmount','sum'),
    Anomalies=('anomaly_v2','sum'),
    Transactions=('anomaly_v2','count')
)

year_stats['Anomaly_Rate'] = (
    year_stats['Anomalies'] /
    year_stats['Transactions']
) * 100

year_stats

,Total_Spend,Anomalies,Transactions,Anomaly_Rate
Year,,,,
2019,73886994.50,739,196806,0.375497
2020,70939105.50,951,179373,0.530180
2021,68059922.62,938,142303,0.659157
2022,77081142.78,999,163942,0.609362
2023,48017420.30,315,106097,0.296898


In [52]:
final_df = df[
    [
        'Year',
        'Month',
        'Department',
        'Division',
        'Merchant',
        'TranxDescription',
        'TranxDate',
        'TrnxAmount',
        'anomaly_v2',
        'anomaly_score'
    ]
]

final_df.to_csv(
    'powerbi.csv',
    index=False
)

In [53]:
df['Amount_Zscore_Department'] = (
    df.groupby('Department')['TrnxAmount']
      .transform(lambda x: (x - x.mean()) / x.std())
)

In [55]:
pip install streamlit plotly pandas numpy scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [56]:
import streamlit as st
import pandas as pd

st.set_page_config(
    page_title="Credit Card Risk Dashboard",
    layout="wide"
)

st.title("Credit Card Transaction Risk Analytics Dashboard")

2026-06-05 13:50:34.662 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-05 13:50:34.662 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-05 13:50:34.689 
  command:

    streamlit run /opt/anaconda3/lib/python3.13/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-06-05 13:50:34.689 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [58]:
@st.cache_data
def load_data():
    return pd.read_csv("powerbi.csv")

df = load_data()

2026-06-05 13:51:38.349 No runtime found, using MemoryCacheStorageManager
2026-06-05 13:51:38.351 No runtime found, using MemoryCacheStorageManager
2026-06-05 13:51:38.352 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-05 13:51:38.352 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-05 13:51:38.353 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-05 13:51:38.868 Thread 'Thread-6': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-05 13:51:38.876 Thread 'Thread-6': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-05 13:51:38.876 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-05 13:51:38.876 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored w

In [59]:
st.write(df.head())

2026-06-05 13:51:51.773 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-05 13:51:51.773 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
